In [25]:
import pandas as pd

In [26]:
df = pd.read_excel("C:/Users/IBRA/Desktop/online-retail-sales-analysis/data/raw/Online Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [27]:
df_clean = df.copy()
df_clean.shape

(541909, 8)

## Removing Duplicate Rows

Duplicate rows can affect the accuracy of sales, product, and customer analysis because they may cause transactions to be counted more than once.

The duplicate rows will be removed from the cleaned copy of the dataset, while the original raw dataset will remain unchanged.

In [28]:
df_clean.duplicated().sum()

np.int64(5268)

In [29]:
df_clean = df_clean.drop_duplicates()
df_clean.shape

(536641, 8)

In [30]:
df_clean.duplicated().sum()

np.int64(0)

### Duplicate Cleaning Result

The dataset originally contained 5,268 duplicate rows.

After removing duplicate rows, the dataset was reduced from 541,909 rows to 536,641 rows.

A final duplicate check confirmed that there are now 0 duplicated rows in the cleaned dataset.

## Identifying Cancelled Transactions

In the Online Retail dataset, invoice numbers that begin with `C` represent cancelled transactions.

Before removing these rows, a cancellation flag will be created. This makes it easier to compare cancelled transactions with negative quantities and decide how they should be handled.

In [31]:
df_clean['IsCancelled'] = df_clean['InvoiceNo'].astype(str).str.startswith('C')

df_clean['IsCancelled'].sum()

np.int64(9251)

In [32]:
(df_clean['Quantity'] < 0).sum()

np.int64(10587)

In [33]:
((df_clean['IsCancelled'] == True) & (df_clean['Quantity'] < 0)).sum()

np.int64(9251)

### Cancelled and Negative Quantity Observations

After removing duplicate rows, the dataset contains 9,251 cancelled transaction rows where `InvoiceNo` starts with `C`.

The dataset also contains 10,587 rows with negative quantities.

All 9,251 cancelled transaction rows also have negative quantities. However, there are 1,336 additional rows with negative quantities that are not marked as cancelled invoices.

For the cleaned sales dataset, cancelled transactions and remaining negative quantity rows will be removed because the main analysis focuses on completed positive sales transactions.

## Removing Cancelled and Negative Quantity Transactions

The main goal of this project is to analyze completed sales transactions.

Cancelled invoices and rows with negative quantities do not represent completed positive sales, so they will be removed from the cleaned sales dataset.

In [34]:
df_clean = df_clean[(df_clean['IsCancelled'] == False) & (df_clean['Quantity'] > 0)]

df_clean.shape

(526054, 9)

In [35]:
df_clean['IsCancelled'].sum(), (df_clean['Quantity'] < 0).sum()

(np.int64(0), np.int64(0))

### Cancelled and Negative Quantity Cleaning Result

After removing cancelled transactions and rows with negative quantities, the cleaned dataset contains 526,054 rows and 9 columns.

A validation check confirmed that there are now:

- 0 cancelled invoice rows
- 0 rows with negative quantities

The dataset is now closer to representing completed positive sales transactions.

## Handling Zero or Negative Unit Prices

For completed sales analysis, each transaction should have a positive unit price.

Rows where `UnitPrice` is less than or equal to zero may represent free items, test entries, adjustments, or invalid sales records. These rows will be removed from the cleaned sales dataset.

In [36]:
(df_clean['UnitPrice'] <= 0).sum()

np.int64(1176)

### Zero or Negative Unit Price Check

After removing duplicates, cancelled invoices, and negative quantity rows, the dataset still contains 1,176 rows where `UnitPrice` is less than or equal to zero.

For completed sales analysis, these rows will be removed because valid sales transactions should have a positive quantity and a positive unit price.

In [37]:
df_clean = df_clean[df_clean['UnitPrice'] > 0]

df_clean.shape

(524878, 9)

In [38]:
(df_clean['UnitPrice'] <= 0).sum()

np.int64(0)

### Unit Price Cleaning Result

After removing rows where `UnitPrice` is less than or equal to zero, the cleaned dataset contains 524,878 rows and 9 columns.

A validation check confirmed that there are now 0 rows with zero or negative unit prices.

At this stage, the cleaned dataset only contains rows with positive quantities and positive unit prices, which is suitable for completed sales analysis.

In [39]:
df_clean.isnull().sum()

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132186
Country             0
IsCancelled         0
dtype: int64

## Handling Missing Customer IDs

After the previous cleaning steps, the only remaining missing values are in the `CustomerID` column.

The cleaned dataset contains 132,186 rows with missing `CustomerID` values.

These rows will not be removed from the main cleaned dataset because they can still be useful for sales, product, country, and invoice-level analysis.

However, rows with missing `CustomerID` will be excluded later from customer-level analysis because customer behavior cannot be analyzed without a customer identifier.

In [40]:
df_clean['HasCustomerID'] = df_clean['CustomerID'].notna()

df_clean[['CustomerID', 'HasCustomerID']].head()

,CustomerID,HasCustomerID
0,17850.0,True
1,17850.0,True
2,17850.0,True
3,17850.0,True
4,17850.0,True


In [41]:
df_clean['HasCustomerID'].value_counts()

HasCustomerID
True     392692
False    132186
Name: count, dtype: int64

### Customer ID Flag Result

A new column called `HasCustomerID` was created to identify whether each row has a customer identifier.

The cleaned dataset contains:

- 392,692 rows with a `CustomerID`
- 132,186 rows without a `CustomerID`

Rows without a `CustomerID` will remain in the main cleaned dataset because they are still useful for sales, product, country, and invoice-level analysis. However, they will be excluded later from customer-level analysis.

## Checking Product Description Consistency

Each `StockCode` should ideally have one consistent product description.

If the same `StockCode` has multiple descriptions, this may cause problems during product analysis because the same product could appear under different names.

In [42]:
description_counts = (
    df_clean
    .groupby('StockCode')['Description']
    .nunique()
    .sort_values(ascending=False)
)

(description_counts > 1).sum()

np.int64(220)

### Product Description Consistency Result

After cleaning duplicate rows, cancelled transactions, invalid quantities, and invalid prices, 220 `StockCode` values still have more than one product description.

This means some products may appear under slightly different names. To make product analysis cleaner, a standardized product description will be created using the most common description for each `StockCode`.

In [43]:
description_counts[description_counts > 1].head(10)

StockCode
23196     4
23236     4
23396     3
23126     3
23535     3
23209     3
23370     3
23240     3
23366     3
17107D    3
Name: Description, dtype: int64

In [44]:
conflicting_codes = description_counts[description_counts > 1].index

df_clean[df_clean['StockCode'].isin(conflicting_codes)] \
    .groupby('StockCode')['Description'] \
    .unique() \
    .head(10)

StockCode
20622            [VIPPASSPORT COVER , VIP PASSPORT COVER ]
20725      [LUNCH BAG RED RETROSPOT, LUNCH BAG RED SPOTTY]
20914    [SET/5 RED RETROSPOT LID GLASS BOWLS, SET/5 RE...
21109    [LARGE CAKE TOWEL, CHOCOLATE SPOTS, LARGE CAKE...
21112    [SWISS ROLL TOWEL, PINK  SPOTS, SWISS ROLL TOW...
21175    [GIN + TONIC DIET METAL SIGN, GIN AND TONIC DI...
21232    [STRAWBERRY CERAMIC TRINKET BOX, STRAWBERRY CE...
21243        [PINK  POLKADOT PLATE , PINK POLKADOT PLATE ]
21507    [ELEPHANT, BIRTHDAY CARD, , ELEPHANT BIRTHDAY ...
21811    [CHRISTMAS HANGING HEART WITH BELL, HANGING HE...
Name: Description, dtype: object

In [45]:
description_mapping = (
    df_clean
    .groupby('StockCode')['Description']
    .agg(lambda x: x.value_counts().idxmax())
)

df_clean['ProductDescription'] = df_clean['StockCode'].map(description_mapping)

df_clean[['StockCode', 'Description', 'ProductDescription']].head()

,StockCode,Description,ProductDescription
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,WHITE HANGING HEART T-LIGHT HOLDER
1,71053,WHITE METAL LANTERN,WHITE METAL LANTERN
2,84406B,CREAM CUPID HEARTS COAT HANGER,CREAM CUPID HEARTS COAT HANGER
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,KNITTED UNION FLAG HOT WATER BOTTLE
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,RED WOOLLY HOTTIE WHITE HEART.


In [46]:
df_clean.groupby('StockCode')['ProductDescription'].nunique().max()

1

### Product Description Standardization Result

Before standardization, 220 `StockCode` values had more than one product description.

A new column called `ProductDescription` was created using the most common description for each `StockCode`.

The original `Description` column was kept for reference, while `ProductDescription` will be used later for product-level analysis.

A validation check confirmed that each `StockCode` now maps to only one standardized product description.

## Creating Total Price

A new column called `TotalPrice` will be created by multiplying `Quantity` by `UnitPrice`.

This column represents the total sales value for each transaction row and will be used later for revenue, product, country, and customer analysis.

In [47]:
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

df_clean[['Quantity', 'UnitPrice', 'TotalPrice']].head()

,Quantity,UnitPrice,TotalPrice
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


In [48]:
df_clean['TotalPrice'].describe()

count    524878.000000
mean         20.275399
std         271.693566
min           0.001000
25%           3.900000
50%           9.920000
75%          17.700000
max      168469.600000
Name: TotalPrice, dtype: float64

### Total Price Result

The `TotalPrice` column was successfully created by multiplying `Quantity` by `UnitPrice`.

The minimum `TotalPrice` is positive, which confirms that the dataset now contains only positive sales values.

However, the maximum `TotalPrice` is 168,469.60, which is much higher than the average transaction row value. This may represent a valid bulk purchase or an outlier transaction, so it should be investigated rather than removed automatically.

In [49]:
df_clean.sort_values('TotalPrice', ascending=False)[
    ['InvoiceNo', 'StockCode', 'ProductDescription', 'Quantity', 'UnitPrice', 'TotalPrice', 'Country']
].head(10)

,InvoiceNo,StockCode,ProductDescription,Quantity,UnitPrice,TotalPrice,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,United Kingdom
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,United Kingdom
222680,556444,22502,PICNIC BASKET WICKER SMALL,60,649.50,38970.00,United Kingdom
15017,537632,AMAZONFEE,AMAZON FEE,1,13541.33,13541.33,United Kingdom
299982,A563185,B,Adjust bad debt,1,11062.06,11062.06,United Kingdom
173382,551697,POST,POSTAGE,1,8142.75,8142.75,United Kingdom
348325,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,5.06,7144.72,United Kingdom
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40,United Kingdom
160546,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40,United Kingdom
421601,573003,23084,RABBIT NIGHT LIGHT,2400,2.08,4992.00,Netherlands


In [50]:
Q1 = df_clean['TotalPrice'].quantile(0.25)
Q3 = df_clean['TotalPrice'].quantile(0.75)
IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR

df_clean['IsHighValueOutlier'] = df_clean['TotalPrice'] > upper_limit

upper_limit, df_clean['IsHighValueOutlier'].sum()

(np.float64(38.400000000000006), np.int64(42624))

### High-Value Transaction Observations

The highest `TotalPrice` rows show a mix of transaction types.

Some rows appear to be valid bulk product purchases, such as large quantities of regular products. However, some rows are not normal product sales, such as `AMAZONFEE`, `POSTAGE`, and `Adjust bad debt`.

Using the IQR method, the upper limit for high-value rows is 38.40, and 42,624 rows are flagged as high-value outliers.

Because retail transaction data is naturally skewed, these rows should not be removed automatically. Instead, they will be flagged using the `IsHighValueOutlier` column for later analysis.

## Checking Non-Product Transaction Codes

Some rows appear to represent non-product transactions, such as postage, fees, manual adjustments, or bad debt.

These rows may be valid accounting records, but they should be identified because they may distort product-level analysis and market basket analysis.

In [51]:
df_clean['StockCode'] = df_clean['StockCode'].astype(str)

special_stockcodes = df_clean[
    df_clean['StockCode'].str.contains(r'^[A-Za-z]+$', regex=True)
]

special_stockcodes[['StockCode', 'ProductDescription']].drop_duplicates().sort_values('StockCode')

,StockCode,ProductDescription
15017,AMAZONFEE,AMAZON FEE
299982,B,Adjust bad debt
84016,DCGSSBOY,BOYS PARTY BAG
84017,DCGSSGIRL,GIRLS PARTY BAG
1814,DOT,DOTCOM POSTAGE
2239,M,Manual
157195,PADS,PADS TO MATCH ALL CUSHIONS
45,POST,POSTAGE
152709,S,SAMPLES
40383,m,Manual


In [52]:
special_stockcodes['StockCode'].value_counts()

StockCode
POST         1126
DOT           706
M             316
DCGSSGIRL      13
DCGSSBOY       11
PADS            3
AMAZONFEE       2
S               2
m               1
B               1
Name: count, dtype: int64

### Non-Product Code Observations

Several stock codes appear to represent non-product transaction records rather than regular retail products.

Examples include postage, manual adjustments, Amazon fees, bad debt adjustments, and samples.

These rows may be valid business records, but they can distort product-level analysis and market basket analysis. Therefore, they will be flagged using a new column called `IsNonProduct`.

The rows will not be deleted from the main cleaned dataset. Instead, the flag will allow product-level analysis to exclude them later when needed.

In [53]:
non_product_codes = ['AMAZONFEE', 'B', 'DOT', 'M', 'm', 'POST', 'S']

df_clean['IsNonProduct'] = df_clean['StockCode'].isin(non_product_codes)

df_clean['IsNonProduct'].value_counts()

IsNonProduct
False    522724
True       2154
Name: count, dtype: int64

In [54]:
df_clean[df_clean['IsNonProduct'] == True] \
    .groupby(['StockCode', 'ProductDescription']) \
    .size() \
    .sort_values(ascending=False)

StockCode  ProductDescription
POST       POSTAGE               1126
DOT        DOTCOM POSTAGE         706
M          Manual                 316
AMAZONFEE  AMAZON FEE               2
S          SAMPLES                  2
B          Adjust bad debt          1
m          Manual                   1
dtype: int64

### Non-Product Flag Result

A new column called `IsNonProduct` was created to identify transaction rows that do not represent normal retail products.

The cleaned dataset contains:

- 522,724 regular product rows
- 2,154 non-product transaction rows

The flagged non-product rows include postage, dotcom postage, manual adjustments, Amazon fees, samples, and bad debt adjustments.

These rows will remain in the cleaned dataset, but they can be excluded later from product-level analysis and market basket analysis when needed.

## Final Validation Checks

Before saving the cleaned dataset, final validation checks will be performed to confirm that the main cleaning issues have been handled correctly.

In [55]:
df_clean.shape

(524878, 14)

In [56]:
df_clean.isnull().sum()

InvoiceNo                  0
StockCode                  0
Description                0
Quantity                   0
InvoiceDate                0
UnitPrice                  0
CustomerID            132186
Country                    0
IsCancelled                0
HasCustomerID              0
ProductDescription         0
TotalPrice                 0
IsHighValueOutlier         0
IsNonProduct               0
dtype: int64

In [57]:
df_clean.duplicated().sum()

np.int64(0)

In [58]:
df_clean['IsCancelled'].sum()

np.int64(0)

In [59]:
(df_clean['Quantity'] <= 0).sum()

np.int64(0)

In [60]:
(df_clean['UnitPrice'] <= 0).sum()

np.int64(0)

In [61]:
(df_clean['TotalPrice'] <= 0).sum()

np.int64(0)

In [62]:
df_clean.dtypes

InvoiceNo                     object
StockCode                     object
Description                   object
Quantity                       int64
InvoiceDate           datetime64[ns]
UnitPrice                    float64
CustomerID                   float64
Country                       object
IsCancelled                     bool
HasCustomerID                   bool
ProductDescription            object
TotalPrice                   float64
IsHighValueOutlier              bool
IsNonProduct                    bool
dtype: object

## Fixing CustomerID Data Type

`CustomerID` is currently stored as a float because the column contains missing values.

Since `CustomerID` is an identifier rather than a numeric measurement, it will be converted to pandas' nullable integer type `Int64`. This keeps missing values while removing unnecessary decimal formatting.

In [63]:
df_clean['CustomerID'] = df_clean['CustomerID'].astype('Int64')

df_clean.dtypes

InvoiceNo                     object
StockCode                     object
Description                   object
Quantity                       int64
InvoiceDate           datetime64[ns]
UnitPrice                    float64
CustomerID                     Int64
Country                       object
IsCancelled                     bool
HasCustomerID                   bool
ProductDescription            object
TotalPrice                   float64
IsHighValueOutlier              bool
IsNonProduct                    bool
dtype: object

## Final Cleaning Summary

The data cleaning process prepared the dataset for analysis while keeping the original raw dataset unchanged.

The main cleaning steps completed were:

- Removed duplicate rows.
- Identified cancelled transactions using invoice numbers starting with `C`.
- Removed cancelled transactions and rows with negative quantities.
- Removed rows with zero or negative unit prices.
- Kept rows with missing `CustomerID` values for general sales, product, country, and invoice-level analysis.
- Created a `HasCustomerID` flag to separate rows with and without customer identifiers.
- Standardized product descriptions using the most common description for each `StockCode`.
- Created a `TotalPrice` column.
- Flagged high-value transaction rows using the IQR method.
- Flagged non-product transaction codes such as postage, manual adjustments, Amazon fees, samples, and bad debt adjustments.
- Converted `CustomerID` to a nullable integer type.

The cleaned dataset is now ready for exploratory data analysis.

In [68]:
df_clean.to_csv("C:/Users/IBRA/Desktop/online-retail-sales-analysis/data/cleaned/online_retail_cleaned.csv", index=False)

In [69]:
pd.read_csv("C:/Users/IBRA/Desktop/online-retail-sales-analysis/data/cleaned/online_retail_cleaned.csv").head()

C:\Users\IBRA\AppData\Local\Temp\ipykernel_17800\4039975519.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv("C:/Users/IBRA/Desktop/online-retail-sales-analysis/data/cleaned/online_retail_cleaned.csv").head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancelled,HasCustomerID,ProductDescription,TotalPrice,IsHighValueOutlier,IsNonProduct
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,True,WHITE HANGING HEART T-LIGHT HOLDER,15.30,False,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,True,WHITE METAL LANTERN,20.34,False,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,True,CREAM CUPID HEARTS COAT HANGER,22.00,False,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,True,KNITTED UNION FLAG HOT WATER BOTTLE,20.34,False,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,True,RED WOOLLY HOTTIE WHITE HEART.,20.34,False,False
